In [1]:
# ============================================================
# EXPERIMENT 9: JSON GENERATION AND VALIDATION
# Groq API + Google Colab
# ============================================================

!pip install -q groq

import os
import json
from getpass import getpass
from groq import Groq


# ------------------------------------------------------------
# STEP 1: Load Groq API Key
# ------------------------------------------------------------

groq_api_key = getpass("Enter your Groq API key: ")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not provided.")

os.environ["GROQ_API_KEY"] = groq_api_key

print("✅ Groq API key loaded successfully.")


# ------------------------------------------------------------
# STEP 2: Create Groq Client
# ------------------------------------------------------------

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("✅ Groq client initialized.")


# ------------------------------------------------------------
# STEP 3: Define LLM Function
# ------------------------------------------------------------

def ask_llm(prompt):

    try:

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0,
            max_tokens=500,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content.strip()

    except Exception as e:

        return f"ERROR: {e}"


# ============================================================
# STEP 4: Initial JSON Prompt
# ============================================================

initial_prompt = """
Generate a JSON array containing exactly 3 books.

Each object MUST contain these keys:
"title", "author", "year"

Requirements:
- Use valid JSON.
- Use double quotes.
- "year" must be a number.
- Do not use trailing commas.
- Output ONLY the JSON.
- Do not include markdown or explanations.

Example format:
[
  {
    "title": "Book Title",
    "author": "Author Name",
    "year": 2000
  }
]
"""


# ============================================================
# STEP 5: Send Initial Prompt
# ============================================================

print("\n" + "=" * 70)
print("INITIAL JSON GENERATION")
print("=" * 70)

print("\nPrompt:")
print(initial_prompt)

response_text = ask_llm(initial_prompt)

print("\nRaw LLM Response:")
print(response_text)


# ============================================================
# STEP 6: Validate JSON
# ============================================================

def validate_books(data):

    errors = []

    # Check array
    if not isinstance(data, list):

        errors.append(
            "Output must be a JSON array."
        )

        return errors

    # Check exactly 3 books
    if len(data) != 3:

        errors.append(
            f"Expected exactly 3 books, found {len(data)}."
        )

    required_keys = {
        "title",
        "author",
        "year"
    }

    # Validate every object
    for i, book in enumerate(data):

        if not isinstance(book, dict):

            errors.append(
                f"Item {i + 1} is not a JSON object."
            )

            continue

        # Check keys
        missing_keys = required_keys - set(book.keys())

        if missing_keys:

            errors.append(
                f"Book {i + 1} is missing keys: "
                f"{missing_keys}"
            )

        # Check year
        if "year" in book:

            if not isinstance(book["year"], int):

                errors.append(
                    f"Book {i + 1}: year must be an integer."
                )

    return errors


# ============================================================
# STEP 7: Parse JSON
# ============================================================

try:

    data = json.loads(response_text)

    print("\n✅ JSON parsing successful.")

    validation_errors = validate_books(data)

except json.JSONDecodeError as e:

    print("\n❌ JSON parsing failed.")
    print("Error:", e)

    data = None
    validation_errors = [
        "Invalid JSON syntax."
    ]


# ============================================================
# STEP 8: Retry with Stricter Prompt if Needed
# ============================================================

if data is None or validation_errors:

    print("\n" + "=" * 70)
    print("RETRYING WITH STRICTER PROMPT")
    print("=" * 70)

    strict_prompt = """
Return ONLY a valid JSON array.

The array MUST contain exactly 3 objects.

Every object MUST contain exactly these three keys:

"title"
"author"
"year"

Rules:
1. Use double quotes around keys and string values.
2. "year" must be an integer.
3. Do not use trailing commas.
4. Do not use Markdown.
5. Do not use ```json.
6. Do not add explanations.
7. Output nothing before or after the JSON array.

Required structure:

[
  {
    "title": "Example",
    "author": "Example Author",
    "year": 2000
  },
  {
    "title": "Example",
    "author": "Example Author",
    "year": 2001
  },
  {
    "title": "Example",
    "author": "Example Author",
    "year": 2002
  }
]
"""

    print("\nStrict Prompt:")
    print(strict_prompt)

    response_text = ask_llm(strict_prompt)

    print("\nRetry Response:")
    print(response_text)

    # --------------------------------------------------------
    # Parse retry
    # --------------------------------------------------------

    try:

        data = json.loads(response_text)

        print("\n✅ Retry JSON parsing successful.")

        validation_errors = validate_books(data)

    except json.JSONDecodeError as e:

        print("\n❌ Retry JSON parsing failed.")
        print("Error:", e)

        data = None

        validation_errors = [
            "Retry produced invalid JSON."
        ]


# ============================================================
# STEP 9: Final Validation
# ============================================================

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

if data is not None and not validation_errors:

    print("✅ Valid JSON.")
    print("✅ Exactly 3 book objects.")
    print("✅ Required keys are present.")
    print("✅ Year values are integers.")


    # Pretty print JSON
    print("\nFinal JSON:")
    print(
        json.dumps(
            data,
            indent=4,
            ensure_ascii=False
        )
    )

else:

    print("❌ Validation failed.")

    for error in validation_errors:

        print(" -", error)


# ============================================================
# STEP 10: Optional YAML Generation
# ============================================================

print("\n" + "=" * 70)
print("OPTIONAL YAML")
print("=" * 70)

try:

    import yaml

    if data is not None and not validation_errors:

        yaml_output = yaml.dump(
            data,
            sort_keys=False,
            allow_unicode=True
        )

        print(yaml_output)

except ImportError:

    print(
        "PyYAML is not installed. "
        "Run: !pip install pyyaml"
    )


# ============================================================
# STEP 11: Final Summary
# ============================================================

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print("""
1. The LLM was asked to generate structured book information.
2. The response was parsed using json.loads().
3. The JSON structure was validated.
4. Required keys were checked.
5. The number of book objects was checked.
6. The year data type was checked.
7. If validation failed, a stricter prompt was used.
8. The validated JSON can be safely used by a Python program.

Conclusion:
Explicit JSON instructions combined with programmatic validation
help produce reliable structured output from an LLM.
""")

print("✅ Experiment 9 completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.1 MB/s eta 0:00:00
Enter your Groq API key: ··········
✅ Groq API key loaded successfully.
✅ Groq client initialized.

INITIAL JSON GENERATION

Prompt:

Generate a JSON array containing exactly 3 books.

Each object MUST contain these keys:
"title", "author", "year"

Requirements:
- Use valid JSON.
- Use double quotes.
- "year" must be a number.
- Do not use trailing commas.
- Output ONLY the JSON.
- Do not include markdown or explanations.

Example format:
[
  {
    "title": "Book Title",
    "author": "Author Name",
    "year": 2000
  }
]


Raw LLM Response:
[
  {
    "title": "To Kill a Mockingbird",
    "author": "Harper Lee",
    "year": 1960
  },
  {
    "title": "1984",
    "author": "George Orwell",
    "year": 1949
  },
  {
    "title": "Pride and Prejudice",
    "author": "Jane Austen",
    "year": 1813
  }
]

✅ JSON parsing successful.

FINAL VALIDATION
✅ Valid JSON.
✅ Exactly 3 book objects.
✅ Required keys are pr